# Week 3 exercises: Loading Data, Databases, APIs, and Joining & Reshaping

*Complete all exercises in this notebook during week 3. The exercises are not submitted — this is purely for your own learning and for preparing for the assignments, project and exam.*

Note! You should always complete the tasks first without using AI. If you need AI to get through these tasks, then you will not be able to pass the oral exam. AI can be used to check if your code could be simplified or improved after you solve the task.

---

## Part 1: Storing API Keys Securely

### Exercise 1: Creating and using a .env file

When working with APIs that require authentication, you need an API key. API keys must **never** be written directly in your code or notebook, because:
- If you upload your notebook to GitHub, anyone can see and [misuse your key](https://www.theregister.com/2026/03/03/gemini_api_key_82314_dollar_charge/).
- Sharing your notebook with a classmate would also expose the key (what if they accidentally share it on Github?).

A simple and safe approach is to store your key in a separate `.env` (env for environment) file that is **excluded from version control**.

a) Create a file called `.env` in the same folder as this notebook (you can do this with a text editor, VS Code, or from Python). I recommend [Sublime Text](https://www.sublimetext.com/) as a convenient, light and free way to view and create practically any file needed in software development. The file should have the following content:

```
# .env (local only, never commit)
GEMINI_API_KEY=YOUR_API_KEY_HERE
```

Replace `YOUR_API_KEY_HERE` with your actual API key. For this exercise, we will use Google's Gemini API as an example (you can obtain one from [Google AI Studio](https://aistudio.google.com/apikey)).

b) Add `.env` to your `.gitignore` file so that the key is never uploaded to GitHub. You can do this in GitHub Desktop by right clicking the file and selecting Ignore File, which will add the file to your .gitignore file. Alternatively, you can manually open (or create) `.gitignore` in a text editor and adding a line containing `.env`. **Do not skip this step.**

## Exercise 2: Reading the API key from the .env file

The `python-dotenv` package automates reading variables that store information such as API keys from the .env file and is the standard approach in Python projects. It loads all variables from a `.env` file into the environment in a single call.

Run the code below to make sure you have stored the API key correctly.

**Note:** You may need to install the package first with `pip install python-dotenv`. 

In [2]:
!pip install python-dotenv

In [4]:
from dotenv import load_dotenv
import os

# Load all variables from .env into the environment
load_dotenv()

api_key = os.environ.get("API_KEY")

if api_key:
    print(f"Key loaded: {api_key[:5]}...")
else:
    print("ERROR: GEMINI_API_KEY not found. Check your .env file.")

Key loaded: 8a288...


## Part 2: Loading and Exporting Data

### Exercise 3: Loading CSV data

The file `data/cafe_sales.csv` contains sales data for a small café.

a) Load the file into a pandas DataFrame. Print the shape and the first 5 rows.

b) Load only the columns `product`, `category`, and `revenue` using the `usecols` parameter. Set `product` as the index. Print the resulting DataFrame.

c) Export the DataFrame from (b) to a new CSV file called `data/category_revenue.csv` without the index column. Then reload the exported file and print it to verify the export was correct.

In [16]:
import pandas as pd

df = pd.read_csv("data/cafe_sales.csv")
df = pd.DataFrame(df)
print(df.shape)
print(df.head())
print()

#bước usecols này là để tách cột ngay từ bước đọc data
df = pd.read_csv("data/cafe_sales.csv", usecols = ["product","category","revenue"])
df = df.set_index("product")
print(df)
print()

df.to_csv("data/category_revenue.csv", index = False)
reload = pd.read_csv("data/category_revenue.csv")
reload = pd.DataFrame(reload)
print(reload)

(12, 5)
      product category  units_sold  unit_price  revenue
0    Espresso   Coffee         320         3.5   1120.0
1  Cappuccino   Coffee         280         4.5   1260.0
2       Latte   Coffee         250         4.9   1225.0
3   Green Tea      Tea         180         3.2    576.0
4  Chai Latte      Tea         150         4.7    705.0

                 category  revenue
product                           
Espresso           Coffee   1120.0
Cappuccino         Coffee   1260.0
Latte              Coffee   1225.0
Green Tea             Tea    576.0
Chai Latte            Tea    705.0
Croissant          Pastry    760.0
Blueberry Muffin   Pastry    714.0
Cinnamon Roll      Pastry    585.0
Iced Americano     Coffee    840.0
Matcha Latte          Tea    624.0
Bagel              Pastry    560.0
Flat White         Coffee    874.0

   category  revenue
0    Coffee   1120.0
1    Coffee   1260.0
2    Coffee   1225.0
3       Tea    576.0
4       Tea    705.0
5    Pastry    760.0
6    Pastry    71

### Exercise 4: Working with JSON data

The file `data/projects.json` contains project records for a small company. Each project has a list of tasks with estimated hours and a status.

a) Load the file using the `json` module. Print the number of projects and the full first record.

b) Use `pd.json_normalize()` to flatten the nested tasks into a regular DataFrame. The result should have one row per task and include the `project_id` and `title` from the parent record. Print the DataFrame. The documentation for pd.json_normalice() is found [here](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html).

c) Export the flattened tasks DataFrame to a JSON file called `data/tasks_flat.json` using the `records` orientation. Load it again and print it to see that it was saved correctly.

In [60]:
import json as js

with open("data/projects.json") as f:
    data = json.load(f)
print("first project: ", data[0]) #its a string, not df so no need to specify the attributes index (columns)
print()
print("number of project: ", len(data))

print()
df = pd.json_normalize(data, meta = ["project_id","title","manager"], record_path = "tasks")
# record path is the path where we want the records in each rows of the df
# meta is the Fields to use as metadata for each record in resulting table.
display(df)

print()
df.to_json("data/tasks_flat.json", orient = "records", indent = 4) #export

with open("data/tasks_flat.json", "r") as f: #load to recheck
    result = js.load(f)
print(result)

first project:  {'project_id': 'P001', 'title': 'Website Redesign', 'manager': 'Laura', 'tasks': [{'task': 'Wireframing', 'hours': 20, 'status': 'completed'}, {'task': 'Frontend development', 'hours': 60, 'status': 'in_progress'}, {'task': 'User testing', 'hours': 15, 'status': 'not_started'}]}

number of project:  3



,task,hours,status,project_id,title,manager
0,Wireframing,20,completed,P001,Website Redesign,Laura
1,Frontend development,60,in_progress,P001,Website Redesign,Laura
2,User testing,15,not_started,P001,Website Redesign,Laura
3,UI design,35,completed,P002,Mobile App Launch,Jukka
4,Backend API,80,in_progress,P002,Mobile App Launch,Jukka
5,App store submission,5,not_started,P002,Mobile App Launch,Jukka
6,Beta testing,25,not_started,P002,Mobile App Launch,Jukka
7,Schema mapping,15,completed,P003,Data Pipeline Migration,Sara
8,ETL scripts,45,completed,P003,Data Pipeline Migration,Sara
9,Validation,20,in_progress,P003,Data Pipeline Migration,Sara



[{'task': 'Wireframing', 'hours': 20, 'status': 'completed', 'project_id': 'P001', 'title': 'Website Redesign', 'manager': 'Laura'}, {'task': 'Frontend development', 'hours': 60, 'status': 'in_progress', 'project_id': 'P001', 'title': 'Website Redesign', 'manager': 'Laura'}, {'task': 'User testing', 'hours': 15, 'status': 'not_started', 'project_id': 'P001', 'title': 'Website Redesign', 'manager': 'Laura'}, {'task': 'UI design', 'hours': 35, 'status': 'completed', 'project_id': 'P002', 'title': 'Mobile App Launch', 'manager': 'Jukka'}, {'task': 'Backend API', 'hours': 80, 'status': 'in_progress', 'project_id': 'P002', 'title': 'Mobile App Launch', 'manager': 'Jukka'}, {'task': 'App store submission', 'hours': 5, 'status': 'not_started', 'project_id': 'P002', 'title': 'Mobile App Launch', 'manager': 'Jukka'}, {'task': 'Beta testing', 'hours': 25, 'status': 'not_started', 'project_id': 'P002', 'title': 'Mobile App Launch', 'manager': 'Jukka'}, {'task': 'Schema mapping', 'hours': 15, 'st

## Part 3: Databases

### Exercise 5: Connecting and querying a SQLite database

The file `data/retail.db` is a SQLite database with tables including Product, Vendor, Store, Region, Customer, SalesTransaction, and Includes. The tables, relationships and a sample of the data are visualized below:

<img src="data/schema.png" alt="Database schema" width="700">


a) Connect to the database using `sqlite3`. List all table names by querying `sqlite_master`.

b) Using `fetchall()`, retrieve and print the names and prices of all products that cost more than 100€. Format the output so that each product is printed as `ProductName: Price€`.

c) Using a parameterised query (with `?` placeholders), retrieve all products supplied by vendor `"MK"`. Print the product names.

In [139]:
import sqlite3 

conn = sqlite3.connect("data/retail.db")
cursor = conn.cursor()
# sqlite_master — a built-in SQLite table that stores information about the database structure (tables, indexes, views)
cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table'") # the '*' means 'everything'
a = cursor.fetchall()
print(a)
print()

queryb = """
    SELECT productname AS name, productprice AS price
    FROM product
    WHERE price > 100
    """
test = "PRAGMA table_info(product)" #this is for check the info of tables

cursor.execute(queryb)
b = cursor.fetchall()
for name,price in b:      #must loop here to print out a string
    print(f"{name}: {price} euro")

print()
vendorid = "MK"
queryc = """
    SELECT productname
    FROM product
    WHERE vendorid = ?
    """
cursor.execute(queryc,(vendorid,))
c = cursor.fetchall()
print(c)

conn.close()

[('vendor',), ('category',), ('product',), ('region',), ('store',), ('customer',), ('salestransaction',), ('includes',)]

Tiny Tent: 150 euro
Biggy Tent: 250 euro
Hi-Tec GPS: 300 euro
Comfy Harness: 150 euro
Sunny Charger: 125 euro
Mega Camera: 275 euro
Luxo Tent: 500 euro

[('Easy Boot',), ('Cosy Sock',), ('Tiny Tent',), ('Biggy Tent',), ('Power Pedals',), ('Comfy Harness',), ('Strongster Carribeaner',), ('Electra Compass',)]


### Exercise 6: Loading database results into pandas

Using the same `retail.db` database:

a) Use `pd.read_sql_query()` to load the entire `Customer` table into a DataFrame. Print the shape and the first 5 rows.

b) Write a SQL query that joins the `SalesTransaction` and `Customer` tables on `CustomerID` and returns `CustomerName`, `TDate`, and `StoreID`. Load the result into a DataFrame and print the first 5 rows. If you have not taken an introductory database course, use an AI chatbot to solve this part.

c) Close the database connection.

In [138]:
conn = sqlite3.connect("data/retail.db")
cursor = conn.cursor()

df_customer = pd.read_sql_query("SELECT * FROM Customer", conn) #format: ("query", data)
print(df_customer.shape)
print(df_customer.head())

print()
query6b = """
    SELECT CustomerName, TDate, StoreID
    FROM SalesTransaction s
    INNER JOIN Customer c
        ON s.CustomerID = c.CustomerID
    """
df_join = pd.read_sql_query(query6b,conn)
# if use pd.DataFrame(cursor.execute(....).fetchall()), the result is a df of tuples since fetchall returns tuple.
# the column names wont show then. so use read_sql_query so that it reads info directly from there
print(df_join.head())

conn.close()

(10, 3)
  customerid customername customerzip
0    1-2-333         Tina       60137
1    2-3-444         Tony       60611
2    3-4-555          Pam       35401
3    4-5-666         Elly       47374
4    5-6-777         Nora       60640

  customername       tdate storeid
0         Tina  2020-01-01      S1
1         Tony  2020-01-01      S2
2         Tina  2020-01-02      S3
3          Pam  2020-01-02      S3
4         Tony  2020-01-02      S3


## Part 4: APIs

### Exercise 7: Fetching data from a public API

a) Use the `requests` library to fetch the latest EUR exchange rates from:  
`https://open.er-api.com/v6/latest/EUR`

Check that the status code is 200 and print the exchange rates for USD, GBP, and SEK.

b) Convert the full rates dictionary into a pandas DataFrame with columns `currency` and `rate`. Print the first 10 rows.

c) Using the Open-Meteo API, fetch the 7-day forecast for Turku (latitude 60.45, longitude 22.27) with `temperature_2m_max` and `precipitation_sum` as daily variables and `Europe/Helsinki` as the timezone. Convert the result into a DataFrame with columns `date`, `temp_max`, and `precipitation`. Print the DataFrame.

In [140]:
import requests

url = "https://open.er-api.com/v6/latest/EUR"
respond = requests.get(url) #.get() to connect to the url
print(respond.status_code) #.status_code attribute to see the status of the link

print()
if respond.status_code == 200:
    info = respond.json()
    rate = info["rates"]  # first connect to the columns
    print("USD: ",rate["USD"]) #then access the components
    print("GBP: ",rate["GBP"])
    print("SEK: ",rate["SEK"])
else:
    print("Code not responding: ",respond.status_code)

print()
info = respond.json()
dictionary = info["rates"]
df7 = pd.DataFrame(dictionary.items(), columns = ["currency","rate"])
# .items() turns the dictionary into a list of tuples [("USD", 1.08), ("GBP", 0.85), ...]
# columns defines the col names
print(df7.head(10))

print()
url = "https://api.open-meteo.com/v1/forecast?latitude=60.45&longitude=22.27&daily=temperature_2m_max,precipitation_sum&timezone=Europe/Helsinki"
forecast = requests.get(url) #access the api server
forecast_info = forecast.json() #get results as json
select_info = forecast_info["daily"]
df7c = pd.DataFrame(select_info) #convert to a df
print(df7c)

200

USD:  1.169144
GBP:  0.867225
SEK:  10.809107

  currency         rate
0      EUR     1.000000
1      AED     4.293680
2      AFN    74.657724
3      ALL    95.491054
4      AMD   435.360265
5      ANG     2.092767
6      AOA  1101.542703
7      ARS  1622.416237
8      AUD     1.636996
9      AWG     2.092767

         time  temperature_2m_max  precipitation_sum
0  2026-04-24                13.2                0.0
1  2026-04-25                 8.1                0.9
2  2026-04-26                 6.6                0.0
3  2026-04-27                 8.9                0.8
4  2026-04-28                 7.8                0.0
5  2026-04-29                 8.4                0.0
6  2026-04-30                13.6                0.0


### Exercise 8: Using an API key with Gemini

This exercise uses the Gemini API key that you stored in `.env` in Exercise 1.

a) Load your API key from `.env` using either approach from Part 1.

b) Install the `google-genai` package (if needed) and use it to send a simple prompt to Gemini. For example, ask it to explain what an API is in one sentence. Print the response.

```python
from google import genai

client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Explain what an API is in one sentence."
)
print(response.text)
```

c) Write a function called `ask_gemini(prompt)` that takes a prompt string as input, sends it to the Gemini API, and returns the response text. Test it with a prompt of your choice.

In [ ]:
# YOUR CODE HERE


## Part 5: Joining and Reshaping Data

### Exercise 9: Merging DataFrames

Consider the following two DataFrames:

```python
stores = pd.DataFrame({
    "store_id": [1, 2, 3, 4],
    "city": ["Helsinki", "Turku", "Tampere", "Oulu"],
    "region": ["South", "South", "Central", "North"]
})

sales = pd.DataFrame({
    "transaction_id": [1001, 1002, 1003, 1004, 1005],
    "store_id": [1, 2, 2, 3, 5],
    "amount": [250, 120, 310, 180, 90]
})
```

a) Perform an **inner join** on `store_id`. How many rows are in the result and why?

b) Perform a **left join** with `stores` on the left. Which store has no sales and how is it represented?

c) Perform an **outer join**. Identify which rows have `NaN` values and explain why.

In [146]:
stores = pd.DataFrame({
    "store_id": [1, 2, 3, 4],
    "city": ["Helsinki", "Turku", "Tampere", "Oulu"],
    "region": ["South", "South", "Central", "North"]
})

sales = pd.DataFrame({
    "transaction_id": [1001, 1002, 1003, 1004, 1005],
    "store_id": [1, 2, 2, 3, 5],
    "amount": [250, 120, 310, 180, 90]
})

merge1 = pd.merge(stores, sales, on = "store_id", how = "inner")
print(len(merge1))
# the answer is 4 since there are 4 lines that share similar contents in the column joined on

print()
merge2 = pd.merge(stores, sales, how = "left")
print(merge2)
# Oulu store has no sales since its id does not exist in the right dataframe sales

print()
merge3 = pd.merge(stores, sales, how = "outer")
print(merge3)
#lines with id 4 and 5 have NaN values because these ids do not exist in the other's data


4

   store_id      city   region  transaction_id  amount
0         1  Helsinki    South          1001.0   250.0
1         2     Turku    South          1002.0   120.0
2         2     Turku    South          1003.0   310.0
3         3   Tampere  Central          1004.0   180.0
4         4      Oulu    North             NaN     NaN

   store_id      city   region  transaction_id  amount
0         1  Helsinki    South          1001.0   250.0
1         2     Turku    South          1002.0   120.0
2         2     Turku    South          1003.0   310.0
3         3   Tampere  Central          1004.0   180.0
4         4      Oulu    North             NaN     NaN
5         5       NaN      NaN          1005.0    90.0


### Exercise 10: Concatenation and reshaping

a) Create two DataFrames representing monthly sales for two quarters:

```python
q1 = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar"],
    "store_A": [20000, 22000, 21000],
    "store_B": [15000, 16000, 17000]
})

q2 = pd.DataFrame({
    "month": ["Apr", "May", "Jun"],
    "store_A": [23000, 24000, 25000],
    "store_B": [17500, 18000, 19000]
})
```

Concatenate them vertically into a single DataFrame with `ignore_index=True`. Print the result.

b) Use `melt()` to convert the concatenated DataFrame from wide to long format. The `month` column should remain as the identifier, and the store columns should become rows. Use `store` as the variable name and `sales` as the value name. Print the result.

c) Use `pivot()` to convert the long DataFrame back to wide format. Verify that the result matches the original concatenated DataFrame.

In [161]:
q1 = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar"],
    "store_A": [20000, 22000, 21000],
    "store_B": [15000, 16000, 17000]
})

q2 = pd.DataFrame({
    "month": ["Apr", "May", "Jun"],
    "store_A": [23000, 24000, 25000],
    "store_B": [17500, 18000, 19000]
})

union = pd.concat([q1, q2], ignore_index = True) #ignore index from original df, make new index for united df
#must put the df into a list since concat() receive only 1 positional input
print(union)
print()

to_long = pd.melt(union,
                  id_vars = "month",
                  var_name = "stores",
                  value_name = "sales"
                 )
print(to_long)
print()

to_wide = pd.pivot(to_long,
                   index = "month",
                   columns = "stores",
                   values = "sales")
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]

to_wide.columns.name = None
to_wide = to_wide.sort_values("month", key=lambda x: x.map({m: i for i, m in enumerate(month_order)}))
to_wide = to_wide.reset_index()
print(to_wide)

  month  store_A  store_B
0   Jan    20000    15000
1   Feb    22000    16000
2   Mar    21000    17000
3   Apr    23000    17500
4   May    24000    18000
5   Jun    25000    19000

   month   stores  sales
0    Jan  store_A  20000
1    Feb  store_A  22000
2    Mar  store_A  21000
3    Apr  store_A  23000
4    May  store_A  24000
5    Jun  store_A  25000
6    Jan  store_B  15000
7    Feb  store_B  16000
8    Mar  store_B  17000
9    Apr  store_B  17500
10   May  store_B  18000
11   Jun  store_B  19000

  month  store_A  store_B
0   Jan    20000    15000
1   Feb    22000    16000
2   Mar    21000    17000
3   Apr    23000    17500
4   May    24000    18000
5   Jun    25000    19000
